# Notebook for running APEX

In [ ]:
# Standard
import os
import psutil
import glob
import pickle
from typing import List
from collections import defaultdict, OrderedDict, Counter
from timeit import default_timer as timer

# Third party
import torch
from tqdm import tqdm
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem.rdchem import Mol
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem import Draw
from scipy import stats

# APEX
from apex.data.atom_bond_features import Vocabulary
from apex.dataset.csl_dataset import CSLDataset
from apex.dataset.labeled_smiles_dataset import LabeledSmilesDataset
from apex.nn.apex import APEXFactorizedCSL, APEXFactorizer
from apex.nn.encoder import LigandEncoder
from apex.nn.probe import LinearProbe

print(f"CPUs: {os.cpu_count()}")
print(f"Total Memory: {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"CUDA available: {torch.cuda.is_available()}")

## Load APEX models
Load some or all as needed depended on analyses

In [ ]:
# paths to CSLs and model weights from Zenodo
models_path = "weights"
libraries_path = "brics_csl"

In [ ]:
# 12M library scored and annotated
scores_with_properties = pd.read_parquet(os.path.join(libraries_path, 'brics_csl_12M', 'enumerated_and_scored.parquet'))
print(scores_with_properties.shape)

### Load encoder and probe

In [ ]:
%%time

# Register the vocabulary
vocab = {"atom": None, "bond": None}
Vocabulary.register_atom_vocab(vocab["atom"])
Vocabulary.register_bond_vocab(vocab["bond"])

# Load model weights
encoder = LigandEncoder.load(os.path.join(models_path, "encoder.pt"))
probe = LinearProbe.load(os.path.join(models_path, "probe.pt"))

# Print the list of modeled endpoints
endpoints = probe.output_names
print(f"Endpoints: {', '.join(endpoints)}.")

### 10B: Surrogate trained on 1M library, factorizer trained on 12M library and fine-tune on 10B library, searching 10B library

In [ ]:
%%time

# Load CSL
dataset_10B = CSLDataset(
    reaction_df=pd.read_parquet(os.path.join(libraries_path, "brics_csl_10B", "reactions.parquet")), 
    synthon_df=pd.read_parquet(os.path.join(libraries_path, "brics_csl_10B", "synthons.parquet")),
    fast_smiles=True
)
print(f"Number of compounds in the CSL = {len(dataset_10B):,}.")

# Load the factorizer
factorizer_10B = APEXFactorizer.load(os.path.join(models_path, "factorizer_brics_csl_10B.pt"))

# Construct APEX model and move to GPU
apex_10B = APEXFactorizedCSL(
    encoder=encoder,
    probe=probe,
    factorizer=factorizer_10B,
    dataset=dataset_10B,
).to('cuda:0')

apex_10B.load_state_dict(torch.load(os.path.join(models_path, "apex_brics_csl_10B.pt"))["model_state_dict"])

### 12M: Surrogate trained on 1M library, factorizer trained on 12M library, searching 12M library

In [ ]:
%%time 

# Load CSL
dataset_12M = CSLDataset(
    reaction_df=pd.read_parquet(os.path.join(libraries_path, "brics_csl_12M", "reactions.parquet")), 
    synthon_df=pd.read_parquet(os.path.join(libraries_path, "brics_csl_12M", "synthons.parquet")),
    fast_smiles=True
)
print(f"Number of compounds in the CSL = {len(dataset_12M):,}.")

# Load the factorizer
factorizer_12M = APEXFactorizer.load(os.path.join(models_path, "factorizer_brics_csl_12M.pt"))

# Construct APEX model and move to GPU
apex_12M = APEXFactorizedCSL(
    encoder=encoder,
    probe=probe,
    factorizer=factorizer_12M,
    dataset=dataset_12M,
).to('cuda:0')

apex_12M.load_state_dict(torch.load(os.path.join(models_path, "apex_brics_csl_12M.pt"))["model_state_dict"])

### 1M: Surrogate trained on 1M library, factorizer trained on 12M library, searching 1M library

In [ ]:
%%time

# Load CSL
dataset_1M = CSLDataset(
    reaction_df=pd.read_parquet(os.path.join(libraries_path, "brics_csl_1M", "reactions.parquet")), 
    synthon_df=pd.read_parquet(os.path.join(libraries_path, "brics_csl_1M", "synthons.parquet")),
    fast_smiles=True
)
print(f"Number of compounds in the CSL = {len(dataset_1M):,}.")

# Construct APEX model and move to GPU
apex_1M = APEXFactorizedCSL(encoder=encoder, probe=probe, factorizer=factorizer_12M, dataset=dataset_1M).to('cuda:0')

# Run pre-computation to facilitate amortized search
apex_1M.update_library_tensors()

## Accuracy of predicted endpoints

In [ ]:
def get_r2(x, y):
    keep_idx = (~np.isnan(x)) & (~np.isnan(y))
    x_ = x[keep_idx]
    y_ = y[keep_idx]
    r2 = 1 - ((y_ - x_) ** 2).mean() / ((y_ - y_.mean()) ** 2).mean()
    return r2

In [ ]:
# Load compounds from evaluation set
df = pd.read_parquet(os.path.join(libraries_path, "brics_csl_1M", "enumerated_and_scored.parquet"))
df = df.set_index('name')
sample_val_df = df[df.fold == 1].sample(100_000, random_state=42)
sample_val_df['key'] = [apex_1M.dataset.name2key(x) for x in sample_val_df.index]

df.fold.value_counts()

In [ ]:
%%time
# calculate factorized surrogate predictions
with torch.no_grad():
    embeds = apex_1M.predict_embeds_from_keys(sample_val_df['key'].tolist())
    preds = apex_1M.probe(embeds)

factorized_surrogate_preds = preds.to('cpu').numpy()

# prepare mols to calculate surrogate predictions
sd = LabeledSmilesDataset(df=sample_val_df, smiles_column='smiles', property_columns=endpoints)
all_mols = [sd[i] for i in tqdm(range(len(sd)))]

# calculate raw surrogate predictions
batch_size = 128
idx = 0
preds = []
while idx < len(all_mols):
    batch = sd.collate_fn(all_mols[idx:idx+batch_size])
    with torch.no_grad():
        val_mols = batch["mols"].to('cuda:0')
        val_values = batch["values"].to('cuda:0')
        val_embeds = apex_1M.encoder(val_mols)
        preds.append(apex_1M.probe(val_embeds))
    idx += batch_size

preds = torch.cat(preds)

surrogate_preds = preds.to('cpu').numpy()

In [ ]:
r2_df = []

# Get predictions and ground truth
ground_truth_values = sample_val_df[endpoints].values

# Calculate R2
for i, name in enumerate(apex_1M.probe.output_names):
    if name in df.columns:

        (x, y) = (factorized_surrogate_preds[:, i], ground_truth_values[:, i])
        r2 = get_r2(x, y)
        r2_df.append({'name':name, 'R2':r2, 'model':'factorized surrogate'})

        (x, y) = (surrogate_preds[:, i], ground_truth_values[:, i])
        r2 = get_r2(x, y)
        r2_df.append({'name':name, 'R2':r2, 'model':'surrogate'})

r2_df = pd.DataFrame(r2_df)

In [ ]:
r2_plotting_order = list(r2_df.sort_values('R2', ascending=False).query('model == "surrogate"').name)

with sns.plotting_context(font_scale=1, rc={"figure.dpi": 300, "font.size":16, "font.family":"Roboto"}):
    g = sns.catplot(
        data=r2_df, kind="bar",
        y="name", x="R2", hue="model",
        errorbar="sd", palette=sns.color_palette('viridis',n_colors=2), alpha=1, height=10, orient='horizontal', 
        order=r2_plotting_order,
        hue_order=['surrogate', 'factorized surrogate'],
    )
    g.set_xlabels('R-squared')
    g.set_ylabels('Endpoint')

plt.axvline(0.0, lw=1, ls='solid', color='black')
plt.axvline(0.2, lw=1, ls='dashed', color='black')
plt.axvline(0.4, lw=1, ls='dashed', color='black')
plt.axvline(0.6, lw=1, ls='dashed', color='black')
plt.axvline(0.8, lw=1, ls='dashed', color='black')
plt.axvline(1.0, lw=1, ls='dashed', color='black')

# List of labels to make bold
labels_to_bold = [x for x in endpoints if x[:5] == 'score']

# Iterate through each axis in the FacetGrid
for ax in g.axes.flat:
    # Iterate through each tick label on the x-axis
    for label in ax.get_yticklabels():
        if label.get_text() in labels_to_bold:
            label.set_fontweight('bold')


## Timings of APEX search

In [ ]:
%%time

maximize = False
available_score_objectives = [x for x in endpoints if x[:5] == 'score']
print(available_score_objectives)

timing_results = []

constraints_for_timing = {
    'Unconstrained': {},
    'Lipinski Rule of 5': {
        "mol_wt": (None, 500.0),
        "n_hbd": (None, 5.5),
        "n_hba": (None, 10.5),
        "logp": (None, 5.0),
    },
}

# devices = ['cpu','cuda']
devices = ['cuda']

for device in devices:
    for model, size in [(apex_12M, "12M"), (apex_10B, "10B")]:
        model.to(device)
        for objective in available_score_objectives:
                for constraint_name, constraint_set in constraints_for_timing.items():
                    for k in [1_000_000, 100_000, 10_000]:
                        print(device, size, objective, constraint_name)
                        start = timer()
                        
                        # Run the APEX search
                        search_results = model.run_apex_search(
                            k=k,
                            objective=objective,
                            maximize=maximize,
                            constraints=constraint_set,
                        )
                        end = timer()
                        timing_results.append({'k':k, 'db':size, 'time':end-start, 'objective':objective, 'df':search_results, 'constraints':constraint_name, 'device':device})            

In [ ]:
timing_df = pd.DataFrame([{k:v for k,v in r.items() if k in ['time','k','db','device','constraints']} for r in timing_results])
timing_df = timing_df.sort_values('k')
timing_df.groupby(['device','db','constraints','k'])['time'].mean()

## Recall @ k

In [ ]:
def filter_with_constraints(df, constraints):
    df_ = df.copy()
    for constraint_name, (lb, ub) in constraints.items():
        if lb is not None:
            df_ = df_[df_[constraint_name] >= lb]
        if ub is not None:
            df_ = df_[df_[constraint_name] <= ub]
    return df_

In [ ]:
def get_ground_truth_topk(df, constraints, k, objective, ascending):
    df_ = filter_with_constraints(df, constraints)
    topk = df_.sort_values(objective, ascending=ascending).head(k)
    return topk

#### Calculate APEX predictions for entire library

In [ ]:
all_preds = []
batch_size = 512
for i in tqdm(range(0, scores_with_properties.shape[0],batch_size)):
    keys = [apex_12M.dataset.name2key(x) for x in scores_with_properties.name.iloc[i:i+batch_size]]
    with torch.no_grad():
        embeds = apex_12M.predict_embeds_from_keys(keys)
        preds = apex_12M.probe(embeds)
        all_preds.append(preds.detach().cpu())
    i += batch_size

In [ ]:
preds_cat = torch.cat(all_preds)
apex_preds = pd.DataFrame(preds_cat.numpy(), columns=['apex_'+column for column in apex_12M.probe.output_names])
scored_with_apex = pd.concat([scores_with_properties, apex_preds], axis=1)

In [ ]:
%%time
recall_at_k = []
k_ranges = (np.power(10, np.arange(-5, 1+.5, 0.5))*scored_with_apex.shape[0]).astype(int)
k_ranges[len(k_ranges) // 2] = 100_000 # use k=100K to match figure

for i, objective in enumerate(available_score_objectives):
    print(objective)

    sort_obj = list(scored_with_apex.sort_values(objective, ascending=True).name)
    sort_apex = list(scored_with_apex.sort_values('apex_'+objective, ascending=True).name)   

    for k in tqdm(k_ranges):
        for j in [100, 1000, 10000, 100000]:
            apex_recall = len(set(sort_obj[:j]) & set(sort_apex[:k]))/j
            recall_at_k.append({'objective':objective, 'k':k, 'j':j, 'recall':apex_recall})

recall_at_k = pd.DataFrame(recall_at_k)
recall_at_k['k_percent'] = recall_at_k['k']/scored_with_apex.shape[0]
recall_at_k = recall_at_k[recall_at_k.k_percent <= 1]

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(3*4,2*4), dpi=300)

colors = ["Blues_r", "Purples_r", "Greens_r", "Oranges_r", "Reds_r"]
base_colors = ['tab:blue','tab:purple','tab:green','tab:orange','tab:red']

for i, objective in enumerate(available_score_objectives):
    
    (ax_i, ax_j) = [(0,0),(0,1),(0,2),(1,0),(1,1)][i]
    sns.lineplot(recall_at_k[recall_at_k.objective == objective], x='k_percent', y='recall', hue='j', palette=sns.color_palette("Blues_r", n_colors=4), ax=axs[ax_i, ax_j], marker='.', markersize=10)

    axs[ax_i, ax_j].set_title(objective[6:], fontsize=12)
    axs[ax_i, ax_j].set_xlabel('evaluation budget (fraction of library)')
    axs[ax_i, ax_j].set_ylabel('recall of top-$j$')        
    axs[ax_i, ax_j].set_xscale('log')
    axs[ax_i, ax_j].axvline(100_000/scored_with_apex.shape[0], linestyle='dashed', zorder=-1, color='darkgray')
    axs[ax_i, ax_j].text(100_000/scored_with_apex.shape[0]*1.2, 0, "$k$=100000", color='darkgray')

axs = axs.flatten()
axs[-1].axis('off')
plt.tight_layout()

### Constraint satisfaction rates

In [ ]:
%%time

apex_search_results = []

all_constraints = {
    'Unconstrained': {},

    'Veber':{
        "n_rotatable_bonds": (0, 10.5),
        "tpsa": (None, 140),
    },

    'Lipinski Rule of 5': {
        "mol_wt": (None, 500.0),
        "n_hbd": (None, 5.5),
        "n_hba": (None, 10.5),
        "logp": (None, 5.0),
    },
    
    '3/75': {
        "tpsa": (75, None),
        "logp": (None, 3.0),
    },

    'Wager CNS': {
        # minus pKa, logD
        'mol_wt':(None, 360),
        'logp':(None, 3),
        'tpsa':(40, 90),
        'n_hbd':(None, 1.5),
    },
    
    'Astex Ro3':{
        "mol_wt": (None, 300.0),
        "n_hbd": (None, 3.5),
        "n_hba": (None, 3.5),
        "logp": (None, 3.0),
        "n_rotatable_bonds":(None, 3.5),
        "tpsa":(None, 60),
    }

}

k = 100_000
maximize = False

results_recall = []
results_satisfaction = []

for constraint_name, constraint_set in all_constraints.items():
    for objective in available_score_objectives:
    
        # Get ground truth
        top_k = get_ground_truth_topk(df=scores_with_properties, k=k, objective=objective, ascending=True, constraints=constraint_set)
        
        # Run the APEX search
        search_results = apex_12M.run_apex_search(
            k=k,
            objective=objective,
            maximize=maximize,
            constraints=constraint_set,
        )
        
        apex_search_results.append({'constraint_name':constraint_name, 'constraint_set':constraint_set, 'objective':objective, 'apex':search_results})
        
        apex_satisfy_constraints = set(search_results[search_results.apex_constr_value == 0.0].name)
        results_satisfaction.append({'constraints':constraint_name,
                                     "objective":objective,
                                     'random_satisfaction':filter_with_constraints(scores_with_properties, constraint_set).shape[0]/scores_with_properties.shape[0],
                                     'topk_satisfaction':filter_with_constraints(scores_with_properties[scores_with_properties.name.isin(apex_satisfy_constraints)], constraint_set).shape[0]/scores_with_properties[scores_with_properties.name.isin(apex_satisfy_constraints)].shape[0]
                                    })
        
        for j in [top_k.shape[0], int(0.1*k), int(0.01*k), int(0.001*k)]:
            results_recall.append({"objective":objective, "recall":len(set(top_k.head(j).name) & set(search_results.name))/j, "k":k, "j":j, 'constraints':constraint_name})
        
results_recall = pd.DataFrame(results_recall)
results_satisfaction = pd.DataFrame(results_satisfaction)

#### Score distributions

In [ ]:
# pull APEX results from recall experiments
unconstrained_apex_12M = {}
for r in apex_search_results:
    if r['constraint_name'] == 'Unconstrained':
        unconstrained_apex_12M[r['objective']] = scores_with_properties[scores_with_properties.name.isin(r['apex'].name)]

In [ ]:
%%time
fig, axs = plt.subplots(2, 3, dpi=150, figsize=(15,10))
axs = axs.flatten()

for i, target in enumerate(['MET','PARP1','ESR1','F10','DRD2']):    
    sns.histplot(scores_with_properties, x=f'score_{target}', ax=axs[i], element='step', bins=50, stat='density', color='k', label='background distribution (12M library)', cumulative=True, fill=False)
    
    sns.histplot(unconstrained_apex_12M[f"score_{target}"], x=f"score_{target}", ax=axs[i], element='step', bins=50, stat='density', color='tab:blue', label='APEX top 100K from 12M library', cumulative=True, fill=False)
    
    axs[i].set_title(target)
    axs[i].set_xlabel('docking score')
    axs[i].set_ylabel('cumulative probability')
    axs[i].grid(color='lightgray', lw=1)

handles, labels = axs[0].get_legend_handles_labels()
axs[-1].set_axis_off()
axs[-1].legend(handles, labels, loc='center')
axs[-1].axis('off')

#### Combined figure

In [ ]:
def constraint_satisfaction_plot(ax, constraint_labels, palette=base_colors, rotation=0):
    sns.scatterplot(results_satisfaction, x='constraints', y='topk_satisfaction', hue='constraints', palette=palette, ax=ax, hue_order=constraint_labels)
    
    category_to_pos = {label.get_text(): pos for pos, label in zip(ax.get_xticks(), ax.get_xticklabels())}
    
    for i, c in enumerate(constraint_labels):
        # Get the position of the current category and the next one
        current_pos = category_to_pos.get(c)
        gdf = results_satisfaction[results_satisfaction.constraints == c]    
        y_val = gdf.random_satisfaction.iloc[0]
        
        # Calculate the start and end points of the line
        xmin = current_pos - 0.4
        xmax = current_pos + 0.4
        
        # Draw the horizontal line
        ax.hlines(y=y_val, xmin=xmin, xmax=xmax, color='k', linestyle='-', linewidth=2, zorder=-1, label=('baseline library satisfaction' if i == 0 else ""))

    ax.set_xticklabels(ax.get_xticklabels(), rotation=rotation)
    
    plt.legend()
    plt.tight_layout()

In [ ]:
constraint_labels = ['Unconstrained', 'Veber', 'Lipinski Rule of 5', 'Pfizer 3/75', 'Wager CNS', 'Astex Rule of 3']

results_satisfaction['objective'] = results_satisfaction['objective'].replace({'score_PARP1':'PARP1','score_MET':'MET','score_DRD2':'DRD2','score_F10':'F10','score_ESR1':'ESR1'})
results_satisfaction['constraints'] = results_satisfaction['constraints'].replace({'Astex Ro3':'Astex Rule of 3', '3/75':'Pfizer 3/75'})

results_recall['constraints'] = results_recall['constraints'].replace({'Astex Ro3':'Astex Rule of 3', '3/75':'Pfizer 3/75'})
results_recall['objective'] = results_recall['objective'].replace({'score_PARP1':'PARP1','score_MET':'MET','score_DRD2':'DRD2','score_F10':'F10','score_ESR1':'ESR1'})

big_fontsize = 20
fontsize = 16

fig = plt.figure(figsize=(20, 16), dpi=300)

gs_new = fig.add_gridspec(nrows=3, ncols=6, hspace=0.5, wspace=0.8, height_ratios=[1,1,1.2])

# columns 1 and 2 (3 rows each, total of 6 grid units high)
ax1 = fig.add_subplot(gs_new[0, 0:2])
ax2 = fig.add_subplot(gs_new[0, 2:4,], sharey=ax1)
ax3 = fig.add_subplot(gs_new[0, 4:6], sharey=ax1)

ax4 = fig.add_subplot(gs_new[1, 0:2])
ax5 = fig.add_subplot(gs_new[1, 2:4])
ax6 = fig.add_subplot(gs_new[1, 4:6])

# column 3 (2 rows, each spanning 3 grid units high)
ax7 = fig.add_subplot(gs_new[2, 0:3]) # top 3/6 of height
ax8 = fig.add_subplot(gs_new[2, 3:6]) # bottom 3/6 of height

axs = [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]

colors = sns.color_palette("tab10", 6)

for i, c in enumerate(constraint_labels):
    sns.barplot(results_recall[results_recall.constraints == c], x='objective', y='recall', hue='j', palette=sns.light_palette(colors[i], n_colors=5)[-4:][::-1], ax=axs[i], edgecolor='k')
    axs[i].set_ylim((0,0.8))
    axs[i].set_title(c, fontsize=big_fontsize)
    axs[i].set_xlabel('target', fontsize=big_fontsize)
    axs[i].set_ylabel('recall of top-$j$', fontsize=big_fontsize)
    axs[i].legend(fontsize=fontsize)
    
constraint_satisfaction_plot(axs[6], constraint_labels, palette=colors, rotation=30)
axs[6].set_xlabel('constraint set', fontsize=big_fontsize)
axs[6].set_ylabel('top-$k$ constraint satisfaction', fontsize=big_fontsize)
handles, labels = axs[6].get_legend_handles_labels()
desired_labels = ['baseline library satisfaction']
new_handles = [h for h, l in zip(handles, labels) if l in desired_labels]
new_labels = [l for l in labels if l in desired_labels]
axs[6].legend(new_handles, new_labels, fontsize=fontsize)
axs[6].set_xticklabels(axs[6].get_xticklabels(), rotation=25, ha="right")

sns.lineplot(recall_at_k, x='k_percent', y='recall', hue='j', palette=sns.light_palette(colors[0], n_colors=5)[-4:][::-1], ax=axs[7], marker='.', markersize=10, errorbar='sd')
axs[7].set_xlabel('evaluation budget (fraction of library)', fontsize=big_fontsize)
axs[7].set_ylabel('recall of top-$j$', fontsize=big_fontsize)        
axs[7].set_xscale('log')
axs[7].axvline(100_000/scored_with_apex.shape[0], linestyle='dashed', zorder=-1, color='gray')
axs[7].text(100_000/scored_with_apex.shape[0]*1.2, 0, "$k$=100000", color='gray', fontsize=fontsize)
axs[7].legend(fontsize=fontsize)

# figure subheadings
axs[0].set_title('A', loc='left', fontsize=fontsize*2.5, fontweight='bold', pad=15, x=-.2) 
axs[6].set_title('B', loc='left', fontsize=fontsize*2.5, fontweight='bold', pad=15, x=-.2/1.5)
axs[7].set_title('C', loc='left', fontsize=fontsize*2.5, fontweight='bold', pad=15, x=-.2/1.5)

for ax in axs:
    ax.tick_params(axis='both', which='major', labelsize=fontsize)

fig.show()

## Properties of the CSL

In [ ]:
# sample products from CSL for visualization and property calculation
csl = dataset_10B
rng = np.random.default_rng()
keys = [csl.idx2key(i) for i in rng.integers(0, len(csl), (1_000_000,))]
sampled_products = []
for k in tqdm(keys):
    prod_name = csl.key2name(*k)
    prod_smiles = csl.key2smiles(reaction_id=k[0], synthon_ids=k[1])
    sampled_products.append({'name':prod_name, 'smiles':prod_smiles})
sampled_products = pd.DataFrame(sampled_products).drop_duplicates()
sampled_products['rxn'] = sampled_products.name.str.split('____', expand=True).iloc[:,0]


In [ ]:
# calculate properties
descriptor_functions = {
    'mol_wt':Chem.Descriptors.ExactMolWt,
    'num_hbd':Chem.Lipinski.NumHDonors,
    'num_hba':Chem.Lipinski.NumHAcceptors,
    'logp':Chem.Crippen.MolLogP,
    'num_rotatable_bonds':Chem.rdMolDescriptors.CalcNumRotatableBonds,
    'tpsa':Chem.rdMolDescriptors.CalcTPSA,
}

mol_properties = []
for row in tqdm(sampled_products.itertuples()):
    this_properties = {'name':row.name}
    mol = Chem.MolFromSmiles(row.smiles)
    for descriptor_name, descriptor_function in descriptor_functions.items():
        this_properties[descriptor_name] = descriptor_function(mol)
    mol_properties.append(this_properties)

mol_properties = pd.DataFrame(mol_properties)
sampled_products = sampled_products.merge(mol_properties, on='name', how='left')
sampled_products['components'] = sampled_products.rxn.str.slice(start=1, stop=2).astype(int)

In [ ]:
mols = [Chem.MolFromSmiles(m) for m in sampled_products.smiles.sample(20)]
    
Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(300, 300), useSVG=False)

In [ ]:
labels = {
    'mol_wt':'Molecular weight',
    'logp':'logP',    
    'num_rotatable_bonds':'Rotatable bonds',
    'tpsa':'TPSA',    
    'num_hbd':'H-bond donors',
    'num_hba':'H-bond acceptors',
}

with sns.plotting_context(font_scale=1, rc={"figure.dpi": 300, "font.size":16, "font.family":"Roboto"}):
    fig, axs = plt.subplots(3, 2, dpi=150, figsize=(15,10))
    
    axs = axs.flatten()
    
    for i, (k,v)  in enumerate(labels.items()):
        if k in ['num_hbd','num_hba','num_rotatable_bonds']:
            bins='auto'
            discrete=True        
        else:
            bins = 50
            discrete = False
    
        sns.histplot(sampled_products, x=k, ax=axs[i], element='step', bins=bins, discrete=discrete, hue='components', palette=["tab:green", "tab:blue"], stat='density')
        axs[i].set_xlabel(v)
    
    plt.tight_layout()